Iqra Kaularikar | 221A060 | 14

In [1]:
import numpy as np
import random

In [2]:
#GridWorld Environment
class GridWorld:
  def __init__(self, size=4, start=(0,0), goal=(3,3)):
    self.size = size
    self.start = start
    self.goal = goal
    self.start = start
    self.actions = [(0,1), (1,0), (0,-1), (-1,0)] #Right, Down, Left, Up

  def reset(self):
    self.state = self.start
    return self.state

  def step(self, action):
    next_state = (self.state[0] + self.actions[action][0],
                  self.state[1] + self.actions[action][1])

    if 0 <= next_state[0] < self.size and 0 <= next_state[1] < self.size:
      self.state = next_state

    reward = 1 if self.state == self.goal else -0.1
    done = self.state == self.goal
    return self.state, reward, done

2. Monte Carlo Control with Eploring

---

Starts

In [3]:
def monte_carlo_control(env, episodes=5000, gamma=0.9, epsilon=0.1):
    Q = {((i, j), a): 0 for i in range(env.size) for j in range(env.size) for a in range(4)}
    returns = {((i, j), a): [] for i in range(env.size) for j in range(env.size) for a in range(4)}

    for _ in range(episodes):
        state = env.reset()
        episode = []

        while True:
            action = random.choice(range(4)) if random.random() < epsilon else max(range(4), key=lambda a: Q[(state, a)])
            next_state, reward, done = env.step(action)
            episode.append((state, action, reward))
            state = next_state
            if done:
                break

        G = 0
        for state, action, reward in reversed(episode):
          G = gamma * G + reward
          if (state, action) not in [(s, a) for s, a, _ in episode[:-1]]:
            returns[(state, action)].append(G)
            Q[(state, action)] = np.mean(returns[(state, action)])

    return Q

3. SARSA (State-Action-Reward-State-Action)

This is a Temporal Difference (TD) learning algorithm


In [4]:
# Temporal Difference Learning (SARSA)
def sarsa(env, episodes=5000, alpha=0.1, gamma=0.9, epsilon=0.1):
    Q = {((i, j), a): 0 for i in range(env.size) for j in range(env.size) for a in range(4)}

    for _ in range(episodes):
        state = env.reset()
        action = random.choice(range(4)) if random.random() < epsilon else max(range(4), key=lambda a: Q[(state, a)])

        while True:
            next_state, reward, done = env.step(action)
            next_action = random.choice(range(4)) if random.random() < epsilon else max(range(4), key=lambda a: Q[(next_state, a)])

            Q[(state, action)] += alpha * (reward + gamma * Q[(next_state, next_action)] - Q[(state, action)])
            state, action = next_state, next_action
            if done:
                break

    return Q

Q -learning is very similar to SARSA but uses max future Q- value instead of following the policy


In [5]:
# Q-Learning Algorithm
def q_learning(env, episodes=5000, alpha=0.1, gamma=0.9, epsilon=0.1):
    Q = {((i, j), a): 0 for i in range(env.size) for j in range(env.size) for a in range(4)}

    for _ in range(episodes):
        state = env.reset()

        while True:
          action = random.choice(range(4)) if random.random() < epsilon else max(range(4), key=lambda a: Q[(state, a)])
          next_state, reward, done = env.step(action)

          Q[(state, action)] += alpha * (reward + gamma * max(Q[(next_state, a)] for a in range(4)) - Q(state, action))
          state = next_state
          if done:
            break

    return Q

Running the environment

Monte Carlo, SARSA, and Q-learning are trained on the GridWorld enviroment

In [7]:
# Initialize Environment
env = GridWorld()

# Train with Monte Carlo
Q_mc = monte_carlo_control(env)

# Train with SARSA
Q_sarsa = sarsa(env)

# Q-Learning Algorithm (Redefined with fix)
def q_learning(env, episodes=5000, alpha=0.1, gamma=0.9, epsilon=0.1):
    Q = {((i, j), a): 0 for i in range(env.size) for j in range(env.size) for a in range(4)}

    for _ in range(episodes):
        state = env.reset()

        while True:
          action = random.choice(range(4)) if random.random() < epsilon else max(range(4), key=lambda a: Q[(state, a)])
          next_state, reward, done = env.step(action)

          Q[(state, action)] += alpha * (reward + gamma * max(Q[(next_state, a)] for a in range(4)) - Q[(state, action)]) # Corrected line
          state = next_state
          if done:
            break

    return Q

# Train with Q-learning
Q_qlearning = q_learning(env)

6. Sample Q-values

At the end, the script prints sample Q-values to compare the different learning methods:


In [8]:
# Print Sample Q-values
print("Sample Q-values from Monte Carlo:", {k: Q_mc[k] for k in list(Q_mc.keys())[:20]})
print("Sample Q-values from SARSA:", {k: Q_sarsa[k] for k in list(Q_sarsa.keys())[:20]})
print("Sample Q-values from Q-Learning:", {k: Q_qlearning[k] for k in list(Q_qlearning.keys())[:20]})

print("Iqra Kaularrikar | 221A060 | 14")

Sample Q-values from Monte Carlo: {((0, 0), 0): 0, ((0, 0), 1): 0, ((0, 0), 2): 0, ((0, 0), 3): 0, ((0, 1), 0): 0, ((0, 1), 1): 0, ((0, 1), 2): 0, ((0, 1), 3): 0, ((0, 2), 0): 0, ((0, 2), 1): 0, ((0, 2), 2): 0, ((0, 2), 3): 0, ((0, 3), 0): 0, ((0, 3), 1): 0, ((0, 3), 2): 0, ((0, 3), 3): 0, ((1, 0), 0): 0, ((1, 0), 1): 0, ((1, 0), 2): 0, ((1, 0), 3): 0}
Sample Q-values from SARSA: {((0, 0), 0): 0.0987134763414054, ((0, 0), 1): 0.07940693595607184, ((0, 0), 2): 0.0032273593976636372, ((0, 0), 3): -0.006415597851853849, ((0, 1), 0): 0.21395643323280458, ((0, 1), 1): 0.21468654212357685, ((0, 1), 2): -0.015460253344933606, ((0, 1), 3): 0.10864738966595044, ((0, 2), 0): 0.24473923536879935, ((0, 2), 1): 0.3988749884319425, ((0, 2), 2): 0.0989640079618688, ((0, 2), 3): 0.2617079133047588, ((0, 3), 0): -0.010517785634701925, ((0, 3), 1): 0.5356371532944049, ((0, 3), 2): -0.02032064653345029, ((0, 3), 3): 0.005313628768810794, ((1, 0), 0): 0.21111463707915246, ((1, 0), 1): 0.05694119323591895,